# GloFAS Historical Discharge — Download
## Simplified Workflow · Step 1

Downloads GloFAS v4.0 reanalysis discharge data (1980–2022) for the Philippines bounding box
from the Copernicus Emergency Management Service (CEMS) Early Warning Data Store (EWDS).

**What this notebook does**

- Submits one API request per year (43 requests) asynchronously — all at once up to `MAX_INFLIGHT`
- Polls for completion and downloads each year as it finishes (no blocking wait)
- Extracts and consolidates into a single `data.grib` per year
- Resumes automatically on re-run — already-downloaded years are skipped

**Prerequisites**

- `conda activate PHLFlood`
- `.cdsapirc` file in this notebook directory with EWDS credentials  
  (URL: `https://ewds.climate.copernicus.eu/api`)
- `pip install ecmwf-datastores-client tqdm requests` (if not already installed)
- G: Drive mounted (output writes to shared drive)

**Output layout**

```
G:\My Drive\GLOFAS_ImpactFloodForecasting_PHL\data\raw\glofas\historical\
  version_4_0\consolidated\discharge\grib2\area_35_63_4_131\
    1980\ -> data.grib
    1981\ -> data.grib
    ...
    2022\ -> data.grib   <- partial year (v4.0 ends 2022-07-31)
    _state\ -> jobs_state.json   <- keep this file: enables resume
```

**Simplified workflow sequence**

| Step | Notebook | Description |
|------|----------|-------------|
| 1 | **This notebook** | Download GloFAS historical (v4.0, 1979-2025) |
| 2 | NB01 | EVT/POT calibration — fit GPD per GloFAS cell |
| 3 | NB02 | Hazard maps — flood depth TIFFs | Probably merged in NB01
| 4 | NB04 | Impact catalogue and EVT2 fit |   Merged between Nb04 and NB05
| 5 | NB05 | Risk profiles — OEP/AEP exceedance curves |

In [23]:
import os
import json
import time
import random
import zipfile
import shutil
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import requests
from tqdm.auto import tqdm
from ecmwf.datastores import Client as DSClient

In [24]:
# ── Credentials ────────────────────────────────────────────────────────────────
# Expects .cdsapirc in the same directory as this notebook.

rc_path = Path(".") / ".cdsapirc"
assert rc_path.exists(), f".cdsapirc not found at: {rc_path.resolve()}"
os.environ["CDSAPI_RC"]  = str(rc_path.resolve())
os.environ["CDSAPI_URL"] = "https://ewds.climate.copernicus.eu/api"


def parse_cdsapirc(path: Path) -> tuple[str, str]:
    url = key = None
    for line in path.read_text(encoding="utf-8").splitlines():
        s = line.strip()
        if s.startswith("url:"):
            url = s.split(":", 1)[1].strip()
        if s.startswith("key:"):
            key = s.split(":", 1)[1].strip()
    if not url or not key:
        raise RuntimeError(f"Could not parse url/key from {path}")
    token = key.split(":", 1)[1] if ":" in key else key
    return url, token


EWDS_URL, EWDS_TOKEN = parse_cdsapirc(rc_path)
print(f"EWDS endpoint : {EWDS_URL}")
print(f"Token loaded  : {'*' * 8}{EWDS_TOKEN[-4:]}")

EWDS endpoint : https://ewds.climate.copernicus.eu/api
Token loaded  : ********3302


In [25]:
# ── Dataset ────────────────────────────────────────────────────────────────────
DATASET        = "cems-glofas-historical"
SYSTEM_VERSION = "version_4_0"
PRODUCT_TYPE   = "consolidated"
VARIABLE       = "river_discharge_in_the_last_24_hours"
HYDRO_MODEL    = "lisflood"
AREA           = [10, -12, 4, -7]   # [N, W, S, E] — AOI BBOX

# ── Time range ─────────────────────────────────────────────────────────────────
START_YEAR = 1979
END_YEAR   = 2025   # GloFAS v4.0 Historical

# ── Output root ────────────────────────────────────────────────────────────────
OUT_DIR = (
    Path.cwd().parents[1]
    / "data"
    / "raw"
    / "glofas"
    / "historical"
    / SYSTEM_VERSION
    / PRODUCT_TYPE
    / "discharge"
    / "grib2"
    / f"area_{AREA[0]}_{AREA[1]}_{AREA[2]}_{AREA[3]}"
)

# ── Async controls ─────────────────────────────────────────────────────────────
# Historical files are large (~300-500 MB/year compressed); keep inflight low.
MAX_INFLIGHT         = 6
DOWNLOAD_WORKERS     = 3
POLL_SECONDS         = 60   # historical jobs queue longer than forecast
JITTER_SECONDS       = 5
MAX_SUBMIT_RETRIES   = 3
MAX_DOWNLOAD_RETRIES = 3

# ── Resume / disk controls ─────────────────────────────────────────────────────
FORCE     = False   # True -> re-download even if data.grib already exists
KEEP_ZIPS = False   # False -> delete staging ZIPs immediately after extraction

In [26]:
# ── State and staging paths (derived from OUT_DIR) ─────────────────────────────
STATE_DIR  = OUT_DIR / "_state"
ZIPS_DIR   = STATE_DIR / "zips"
STATE_PATH = STATE_DIR / "jobs_state.json"


# ── Path helpers ───────────────────────────────────────────────────────────────

def year_out_path(year: int) -> Path:
    return OUT_DIR / str(year) / "data.grib"


def year_zip_path(year: int) -> Path:
    return ZIPS_DIR / f"{year}.zip"


# ── Logging / utility ──────────────────────────────────────────────────────────

def tlog(msg: str) -> None:
    """Thread-safe log via tqdm so progress bar is not corrupted."""
    try:
        tqdm.write(msg)
    except Exception:
        print(msg, flush=True)


def mb(p: Path) -> float:
    return p.stat().st_size / (1024 * 1024)


def safe_unlink(path: Path) -> None:
    try:
        if path.exists():
            path.unlink()
    except Exception:
        pass


def safe_rmtree(path: Path) -> None:
    try:
        if path.exists():
            shutil.rmtree(path, ignore_errors=True)
    except Exception:
        pass


# ── File validation ────────────────────────────────────────────────────────────

def is_valid_zip(path: Path) -> bool:
    if not path.exists() or path.stat().st_size == 0:
        return False
    if not zipfile.is_zipfile(path):
        return False
    try:
        with zipfile.ZipFile(path, "r") as zf:
            _ = zf.namelist()[:5]
        return True
    except Exception:
        return False


def is_grib_payload(p: Path) -> bool:
    """True if p is a non-empty GRIB file (magic bytes b'GRIB')."""
    if (not p.is_file()) or p.stat().st_size == 0:
        return False
    if p.name.lower().endswith(".idx"):
        return False
    try:
        with open(p, "rb") as f:
            return f.read(4) == b"GRIB"
    except Exception:
        return False


# ── Error classification ───────────────────────────────────────────────────────

def is_job_not_found_error(e: Exception) -> bool:
    if isinstance(e, requests.HTTPError):
        resp = getattr(e, "response", None)
        if resp is not None and getattr(resp, "status_code", None) == 404:
            return True
    msg = str(e).lower()
    return "404" in msg and ("job not found" in msg or "deleted" in msg)


def is_bad_request_400(e: Exception) -> bool:
    msg = str(e).lower()
    if "400" in msg and ("bad request" in msg or "invalid request" in msg):
        return True
    if isinstance(e, requests.HTTPError):
        resp = getattr(e, "response", None)
        if resp is not None and getattr(resp, "status_code", None) == 400:
            return True
    return False


# ── GRIB normalization ─────────────────────────────────────────────────────────

def normalize_year_folder(year_dir: Path) -> Path:
    """
    Ensure year_dir contains exactly one GRIB payload named data.grib.
    If the ZIP extracted multiple GRIB files, concatenate them in sorted order.
    """
    year_dir.mkdir(parents=True, exist_ok=True)
    target = year_dir / "data.grib"

    if is_grib_payload(target):
        for idx in year_dir.rglob("*.idx"):
            safe_unlink(idx)
        return target

    payloads = sorted(p for p in year_dir.rglob("*") if is_grib_payload(p))
    if not payloads:
        raise RuntimeError(f"No GRIB payloads found in: {year_dir}")

    if len(payloads) == 1:
        src = payloads[0]
        if src.resolve() != target.resolve():
            if target.exists():
                safe_unlink(target)
            src.replace(target)
        for idx in year_dir.rglob("*.idx"):
            safe_unlink(idx)
        return target

    # Multiple payloads — concatenate (rare but handled)
    tlog(f"  WARNING: {year_dir.name} has {len(payloads)} GRIB parts; concatenating")
    tmp = year_dir / "data.grib.tmp"
    buf = 128 * 1024 * 1024
    with open(tmp, "wb") as w:
        for part in payloads:
            with open(part, "rb") as r:
                shutil.copyfileobj(r, w, length=buf)
    if target.exists():
        safe_unlink(target)
    tmp.replace(target)
    for part in payloads:
        if part.exists() and part.resolve() != target.resolve():
            safe_unlink(part)
    for idx in year_dir.rglob("*.idx"):
        safe_unlink(idx)
    if not is_grib_payload(target):
        raise RuntimeError(f"normalize_year_folder produced invalid data.grib: {target}")
    return target


def extract_and_normalize(year: int, zip_path: Path) -> None:
    """Extract ZIP into year folder and normalize to a single data.grib."""
    year_dir = OUT_DIR / str(year)
    year_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(year_dir)
    if not KEEP_ZIPS:
        safe_unlink(zip_path)
    normalize_year_folder(year_dir)


# ── API request builder ────────────────────────────────────────────────────────

def build_request(year: int) -> dict:
    return {
        "system_version":     [SYSTEM_VERSION],
        "hydrological_model": [HYDRO_MODEL],
        "product_type":       [PRODUCT_TYPE],
        "variable":           [VARIABLE],
        "hyear":              [str(year)],
        "hmonth":             [f"{m:02d}" for m in range(1, 13)],
        "hday":               [f"{d:02d}" for d in range(1, 32)],
        "data_format":        "grib2",
        "download_format":    "zip",
        "area":               AREA,
    }

In [27]:
def save_state(state: dict) -> None:
    """Atomic write via temp-file rename — safe if kernel crashes mid-write."""
    STATE_PATH.parent.mkdir(parents=True, exist_ok=True)
    tmp = STATE_PATH.with_suffix(".tmp")
    tmp.write_text(json.dumps(state, indent=2), encoding="utf-8")
    tmp.replace(STATE_PATH)


def load_state(years: list[int]) -> dict:
    """Load existing state file or initialise a fresh one."""
    if STATE_PATH.exists():
        state = json.loads(STATE_PATH.read_text(encoding="utf-8"))
    else:
        state = {"years": {}}
    for year in years:
        key = str(year)
        if key not in state["years"]:
            state["years"][key] = {
                "status":     "pending",
                "request_id": None,
                "attempts":   0,
                "last_error": None,
            }
    return state


def reconcile_state(state: dict, years: list[int]) -> None:
    """
    Align state with what is actually on disk.
    Called at startup before the main loop to handle prior-crash residue.
    """
    for year in years:
        key  = str(year)
        rec  = state["years"][key]
        grib = year_out_path(year)
        zp   = year_zip_path(year)

        # Already done and file is healthy
        if not FORCE and is_grib_payload(grib):
            rec["status"] = "done"
            continue

        # State says done but file is missing
        if rec.get("status") == "done" and not is_grib_payload(grib):
            tlog(f"  WARNING: {year} marked done but data.grib missing -> resetting")
            rec["status"]     = "pending"
            rec["request_id"] = None
            continue

        # Mid-flight state with no request_id (crash during submit)
        if rec.get("status") in ("downloading", "ready", "running") and not rec.get("request_id"):
            rec["status"]     = "pending"
            rec["request_id"] = None

        # Stale valid ZIP left from a previous crash -> extract it now
        if rec["status"] != "done" and is_valid_zip(zp):
            try:
                extract_and_normalize(year, zp)
                rec["status"] = "done"
                tlog(f"  Recovered {year} from stale zip")
            except Exception as e:
                safe_unlink(zp)
                tlog(f"  ZIP recovery failed for {year}: {e}")

In [28]:
# ── Create directories ─────────────────────────────────────────────────────────
OUT_DIR.mkdir(parents=True, exist_ok=True)
STATE_DIR.mkdir(parents=True, exist_ok=True)
ZIPS_DIR.mkdir(parents=True, exist_ok=True)

# ── Init EWDS client ───────────────────────────────────────────────────────────
ds = DSClient(url=EWDS_URL, key=EWDS_TOKEN)

# ── Build year list ────────────────────────────────────────────────────────────
years = list(range(START_YEAR, END_YEAR + 1))
print(f"Years to download: {START_YEAR}-{END_YEAR}  ({len(years)} total)")

# ── Load and reconcile state ───────────────────────────────────────────────────
state = load_state(years)
reconcile_state(state, years)
save_state(state)

done_n    = sum(1 for y in years if state["years"][str(y)]["status"] == "done")
pending_n = len(years) - done_n
print(f"Already done : {done_n}")
print(f"Remaining    : {pending_n}")
print(f"State file   : {STATE_PATH}")

Years to download: 1979-2025  (47 total)
Already done : 0
Remaining    : 47
State file   : /Users/silvia/Desktop/personal/rodekruis/GLOFAS_ImpactFloodForecasting_PHL/data/raw/glofas/historical/version_4_0/consolidated/discharge/grib2/area_10_-12_4_-7/_state/jobs_state.json


In [32]:
# ── Worker functions ───────────────────────────────────────────────────────────

def download_year_worker(year: int, request_id: str) -> bool:
    """
    Thread worker: download ZIP for one year, extract, normalize to data.grib.
    Creates its own DSClient instance for thread safety.
    """
    grib = year_out_path(year)
    if is_grib_payload(grib):
        zp = year_zip_path(year)
        if zp.exists() and not KEEP_ZIPS:
            safe_unlink(zp)
        return True

    zp = year_zip_path(year)
    if zp.exists() and not is_valid_zip(zp):
        safe_unlink(zp)

    local_ds = DSClient(url=EWDS_URL, key=EWDS_TOKEN)
    for attempt in range(1, MAX_DOWNLOAD_RETRIES + 1):
        try:
            remote = local_ds.get_remote(request_id)
            remote.download(str(zp))
            if not is_valid_zip(zp):
                raise RuntimeError("downloaded file is not a valid ZIP")
            extract_and_normalize(year, zp)
            return True
        except Exception as e:
            safe_unlink(zp)
            sleep_s = min(120, 10 * attempt) + random.uniform(0, 3)
            tlog(f"  {year}: download attempt {attempt} failed: {e} (retry in {sleep_s:.0f}s)")
            time.sleep(sleep_s)
    return False


def submit_year(year: int) -> str:
    """Submit one year's request to EWDS; update state and return request_id."""
    key = str(year)
    rec = state["years"][key]
    for attempt in range(1, MAX_SUBMIT_RETRIES + 1):
        try:
            remote = ds.submit(DATASET, build_request(year))
            rec["request_id"]   = remote.request_id
            rec["status"]       = "submitted"
            rec["attempts"]     = rec.get("attempts", 0) + 1
            rec["submitted_at"] = time.time()
            rec["last_error"]   = None
            save_state(state)
            tlog(f"  submit {year} -> {remote.request_id}")
            return remote.request_id
        except Exception as e:
            if is_bad_request_400(e):
                tlog(f"  {year}: 400 Bad Request — check API parameters: {e}")
                raise
            sleep_s = min(60, 5 * attempt) + random.uniform(0, 2)
            tlog(f"  submit failed {year} attempt {attempt}: {e} (retry in {sleep_s:.0f}s)")
            time.sleep(sleep_s)
    raise RuntimeError(f"Submit permanently failed: {year}")


def reconcile_deleted_job(year: int) -> str:
    """Handle 404 job-not-found: check disk, mark done or reset to pending."""
    key  = str(year)
    rec  = state["years"][key]
    grib = year_out_path(year)
    zp   = year_zip_path(year)

    if is_grib_payload(grib):
        rec["status"]     = "done"
        rec["last_error"] = "job_deleted_but_data_present"
        save_state(state)
        tlog(f"  {year}: job deleted but data.grib exists -> done")
        return "done"

    if is_valid_zip(zp):
        try:
            extract_and_normalize(year, zp)
            rec["status"]     = "done"
            rec["last_error"] = "job_deleted_zip_recovered"
            save_state(state)
            tlog(f"  {year}: job deleted but ZIP recovered -> done")
            return "done"
        except Exception as e:
            safe_unlink(zp)
            tlog(f"  {year}: ZIP recovery failed: {e}")

    rec["status"]     = "pending"
    rec["request_id"] = None
    rec["last_error"] = "job_deleted_resubmit"
    save_state(state)
    tlog(f"  {year}: job deleted, no local data -> will resubmit")
    return "resubmit"


# ── Main two-phase download loop ───────────────────────────────────────────────

inflight: dict[str, str] = {}
ready_queue: list[str]   = []

# Re-register in-flight jobs that survived a previous crash
for year in years:
    key = str(year)
    rec = state["years"][key]
    rid = rec.get("request_id")
    if rid and rec.get("status") in ("submitted", "running", "ready", "downloading"):
        inflight[key] = rid

total = len(years)
pbar  = tqdm(
    total   = total,
    initial = sum(1 for y in years if state["years"][str(y)]["status"] == "done"),
    desc    = "Historical years",
    unit    = "yr",
)

with ThreadPoolExecutor(max_workers=DOWNLOAD_WORKERS) as executor:
    download_futures: dict[str, object] = {}

    def schedule_download(key: str) -> bool:
        if key in download_futures or len(download_futures) >= DOWNLOAD_WORKERS:
            return False
        rec = state["years"][key]
        rid = rec.get("request_id")
        if not rid:
            return False
        rec["status"] = "downloading"
        save_state(state)
        download_futures[key] = executor.submit(download_year_worker, int(key), rid)
        tlog(f"  download queued {key} ({len(download_futures)}/{DOWNLOAD_WORKERS} active)")
        return True

    try:
        while True:
            # 1. Harvest completed downloads
            finished = [k for k, v in download_futures.items() if v.done()]
            for key in finished:
                fut = download_futures.pop(key)
                rec = state["years"][key]
                try:
                    ok = fut.result()
                except Exception as e:
                    tlog(f"  {key}: worker exception: {e}")
                    ok = False
                if ok:
                    rec["status"] = "done"
                    save_state(state)
                    pbar.update(1)
                    p = year_out_path(int(key))
                    tlog(f"  done  {key}  ({mb(p):.1f} MB)")
                else:
                    rec["status"] = "ready"
                    save_state(state)
                    if key not in ready_queue:
                        ready_queue.append(key)
                    tlog(f"  {key}: download failed -> queued for retry")

            # 2. Termination check
            done_count = sum(1 for y in years if state["years"][str(y)]["status"] == "done")
            pbar.n = done_count
            pbar.refresh()
            pbar.set_postfix(
                inflight = len(inflight),
                dl       = len(download_futures),
                queue    = len(ready_queue),
            )
            if done_count >= total and not download_futures:
                break

            progressed = False

            # 3. Drain ready_queue into download pool
            while ready_queue and len(download_futures) < DOWNLOAD_WORKERS:
                key = ready_queue.pop(0)
                if state["years"][key]["status"] == "done":
                    continue
                if schedule_download(key):
                    progressed = True

            # 4. Phase 1: submit pending years up to MAX_INFLIGHT
            for year in years:
                if len(inflight) >= MAX_INFLIGHT:
                    break
                key = str(year)
                rec = state["years"][key]
                if (
                    rec["status"] == "done"
                    or key in inflight
                    or key in download_futures
                    or key in ready_queue
                    or rec["status"] in ("submitted", "running", "ready", "downloading")
                ):
                    continue
                try:
                    rid = submit_year(year)
                    inflight[key] = rid
                    progressed = True
                except Exception as e:
                    tlog(f"  {year}: submit error — skipping this round: {e}")
                    rec["last_error"] = str(e)[:300]
                    save_state(state)

            # 5. Phase 2: poll inflight jobs
            for key in list(inflight.keys()):
                rec = state["years"][key]
                if rec["status"] == "done" or key in download_futures:
                    inflight.pop(key, None)
                    continue
                rid = inflight[key]
                try:
                    remote = ds.get_remote(rid)
                except Exception as e:
                    if is_job_not_found_error(e):
                        rec["last_error"] = str(e)[:300]
                        save_state(state)
                        action = reconcile_deleted_job(int(key))
                        inflight.pop(key, None)
                        progressed = True
                        if action == "done":
                            pbar.update(1)
                    else:
                        tlog(f"  poll failed {key}: {e}")
                        rec["last_error"] = str(e)[:300]
                        save_state(state)
                    continue

                rec["last_poll"] = time.time()
                save_state(state)

                status = getattr(remote, "status", None)
                ready  = getattr(remote, "results_ready", False)

                if status in ("successful", "success") and ready:
                    rec["status"] = "ready"
                    save_state(state)
                    inflight.pop(key, None)
                    if not schedule_download(key):
                        if key not in ready_queue:
                            ready_queue.append(key)
                            tlog(f"  {key}: ready but no download slot -> queued")
                    progressed = True

                elif status in ("failed", "dismissed", "deleted"):
                    tlog(f"  {key}: status={status} -> will resubmit")
                    rec["status"]     = "pending"
                    rec["request_id"] = None
                    save_state(state)
                    inflight.pop(key, None)
                    progressed = True

                else:
                    rec["status"] = "running"
                    save_state(state)

            # 6. Sleep if no progress this round
            if not progressed:
                time.sleep(POLL_SECONDS + random.uniform(0, JITTER_SECONDS))

    finally:
        pbar.close()

done_count = sum(1 for y in years if state["years"][str(y)]["status"] == "done")
print(f"\nDownload complete: {done_count}/{total} years done.")

Historical years:   0%|          | 0/47 [00:00<?, ?yr/s]

  1979: job deleted, no local data -> will resubmit
  1980: job deleted, no local data -> will resubmit
  1981: job deleted, no local data -> will resubmit
  1982: job deleted, no local data -> will resubmit
  1983: job deleted, no local data -> will resubmit
  1984: job deleted, no local data -> will resubmit
  submit 1979 -> 1e9181b1-5d90-4e96-9495-7052273734c8
  submit 1980 -> 15c92028-9b14-4095-89d1-c8198ee3036e
  submit 1981 -> ed9c5f94-3230-43d2-a275-f3630cce6faf
  submit 1982 -> 8c9d66d4-7537-4166-8b47-0f125ab7cb5f
  submit 1983 -> cf31143e-8ad7-458d-975e-28b9dad4e360
  submit 1984 -> df6b5cf9-4339-4e13-86c4-a441e5796647
  download queued 1979 (1/3 active)


60078bb11302df16462270e5b00a76b.zip:   0%|          | 0.00/5.30M [00:00<?, ?B/s]

  submit 1985 -> 69e11709-7cf8-414c-9a0c-ee263a2c44ca
  done  1979  (10.0 MB)
  download queued 1980 (1/3 active)


b650ef08a64aa32f8b84a503086f819e.zip:   0%|          | 0.00/5.40M [00:00<?, ?B/s]

...

  done  2024  (10.0 MB)
  download queued 2025 (1/3 active)


99c1c71eb2a2dc277b6b276477181f70.zip:   0%|          | 0.00/4.41M [00:00<?, ?B/s]

  done  2025  (10.0 MB)
  download queued 2022 (1/3 active)


e63c81c3c3183c2d5fc42a5525a79543.zip:   0%|          | 0.00/4.67M [00:00<?, ?B/s]

  done  2022  (10.0 MB)

Download complete: 47/47 years done.


In [33]:
# ── Verify all years ───────────────────────────────────────────────────────────
print(f"{'Year':>6}  {'OK':^4}  {'Size (MB)':>10}")
print("-" * 26)

failed = []
for year in years:
    p  = year_out_path(year)
    ok = is_grib_payload(p)
    sz = f"{mb(p):.1f}" if p.exists() else "—"
    print(f"  {year}   {'v' if ok else 'X'}   {sz:>10}")
    if not ok:
        failed.append(year)

print()
if failed:
    print(f"WARNING: {len(failed)} year(s) failed GRIB validation — re-run main cell to retry:")
    print(f"  {failed}")
else:
    print(f"All {len(years)} years validated successfully.")

  Year   OK    Size (MB)
--------------------------
  1979   v         10.0
  1980   v         10.0
  1981   v         10.0
  1982   v         10.0
  1983   v         10.0
  1984   v         10.0
  1985   v         10.0
  1986   v         10.0
  1987   v         10.0
  1988   v         10.0
  1989   v         10.0
  1990   v         10.0
  1991   v         10.0
  1992   v         10.0
  1993   v         10.0
  1994   v         10.0
  1995   v         10.0
  1996   v         10.0
  1997   v         10.0
  1998   v         10.0
  1999   v         10.0
  2000   v         10.0
  2001   v         10.0
  2002   v         10.0
  2003   v         10.0
  2004   v         10.0
  2005   v         10.0
  2006   v         10.0
  2007   v         10.0
  2008   v         10.0
  2009   v         10.0
  2010   v         10.0
  2011   v         10.0
  2012   v         10.0
  2013   v         10.0
  2014   v         10.0
  2015   v         10.0
  2016   v         10.0
  2017   v         10.0
  2018   v  